In [1]:
import matplotlib.pyplot as plt
import pickle

from pmbrl.model2 import Model
from pmbrl.data import Experiment_Data, get_data_expanded

In [2]:
nome_do_arquivo = 'regular.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    model = exp['model']

del exp
del arquivo

In [3]:
# data.evaluate_model(model, path='../testing_data(TEST).csv')
data.evaluate_model(model, path='../testing_data.csv')
results = data.get_evaluation_metrics()
results.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,rse_s1,rse_s2,rse_s3,rse_r,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized,rse,rse_normalized
0,0,0,"(0.5280280669895349, 0.1691488904272192)","(-0.018, -0.046, 0.013, -0.019)",0,1.0,"(-0.019, -0.285, 0.013, 1.258)",0.0,1.0,"(-0.025, -0.525, 0.038, 2.534)",...,0.009,0.010,0.468,0.0,0.551214,0.528269,0.525180,0.474715,0.492,0.519845
1,1,0,"(0.5280280669895349, 0.1691488904272192)","(-0.019, -0.285, 0.013, 1.258)",0,1.0,"(-0.025, -0.525, 0.038, 2.534)",1.0,1.0,"(-0.035, -0.288, 0.088, 1.326)",...,0.026,0.025,0.511,0.0,0.557550,0.532448,0.561151,0.478395,0.573,0.532386
2,2,0,"(0.5280280669895349, 0.1691488904272192)","(-0.025, -0.525, 0.038, 2.534)",1,1.0,"(-0.035, -0.288, 0.088, 1.326)",0.0,1.0,"(-0.041, -0.532, 0.115, 2.694)",...,0.056,0.044,1.353,0.0,0.588173,0.539823,0.606715,0.550441,1.493,0.571288
3,3,0,"(0.5280280669895349, 0.1691488904272192)","(-0.035, -0.288, 0.088, 1.326)",0,1.0,"(-0.041, -0.532, 0.115, 2.694)",1.0,1.0,"(-0.052, -0.3, 0.169, 1.601)",...,0.018,0.028,0.812,0.0,0.545935,0.530482,0.568345,0.504150,0.858,0.537228
4,4,0,"(0.5280280669895349, 0.1691488904272192)","(-0.041, -0.532, 0.115, 2.694)",1,1.0,"(-0.052, -0.3, 0.169, 1.601)",0.0,1.0,"(-0.058, -0.546, 0.201, 3.045)",...,0.021,0.014,1.329,0.0,0.582893,0.531219,0.534772,0.548387,1.399,0.549318


In [4]:
expansions = {
    'estimated_p': ['estimated_p0', 'estimated_p1'],
    's': ['s0', 's1', 's2', 's3'],
    's_': ['s_0', 's_1', 's_2', 's_3'],
    's__': ['s__0', 's__1', 's__2', 's__3'],
}


# df = data.evaluation_data[data.evaluation_data['episode'] == 98].copy().reset_index(drop=True)
df = data.evaluation_data.copy()
df = get_data_expanded(df, expansions)
df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s2,s3,s_0,s_1,s_2,s_3,s__0,s__1,s__2,s__3
0,0,0,"(0.5280280669895349, 0.1691488904272192)","(-0.018, -0.046, 0.013, -0.019)",0,1.0,"(-0.019, -0.285, 0.013, 1.258)",0.0,1.0,"(-0.025, -0.525, 0.038, 2.534)",...,0.013,-0.019,-0.019,-0.285,0.013,1.258,-0.025,-0.525,0.038,2.534
1,1,0,"(0.5280280669895349, 0.1691488904272192)","(-0.019, -0.285, 0.013, 1.258)",0,1.0,"(-0.025, -0.525, 0.038, 2.534)",1.0,1.0,"(-0.035, -0.288, 0.088, 1.326)",...,0.013,1.258,-0.025,-0.525,0.038,2.534,-0.035,-0.288,0.088,1.326
2,2,0,"(0.5280280669895349, 0.1691488904272192)","(-0.025, -0.525, 0.038, 2.534)",1,1.0,"(-0.035, -0.288, 0.088, 1.326)",0.0,1.0,"(-0.041, -0.532, 0.115, 2.694)",...,0.038,2.534,-0.035,-0.288,0.088,1.326,-0.041,-0.532,0.115,2.694
3,3,0,"(0.5280280669895349, 0.1691488904272192)","(-0.035, -0.288, 0.088, 1.326)",0,1.0,"(-0.041, -0.532, 0.115, 2.694)",1.0,1.0,"(-0.052, -0.3, 0.169, 1.601)",...,0.088,1.326,-0.041,-0.532,0.115,2.694,-0.052,-0.300,0.169,1.601
4,4,0,"(0.5280280669895349, 0.1691488904272192)","(-0.041, -0.532, 0.115, 2.694)",1,1.0,"(-0.052, -0.3, 0.169, 1.601)",0.0,1.0,"(-0.058, -0.546, 0.201, 3.045)",...,0.115,2.694,-0.052,-0.300,0.169,1.601,-0.058,-0.546,0.201,3.045


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

def find_params(m, inputs, target, initial_params, num_epochs=5000, learning_rate=.01):
    input_values = torch.tensor(inputs).reshape(1, len(inputs))
    param = torch.tensor(initial_params, requires_grad=True)
    target_value = torch.tensor(target).reshape(1, len(target))

    optimizer = optim.Adam([param], lr=learning_rate)
    criterion = nn.MSELoss(reduction='none')

    history = []

    for p in m.parameters():
        p.requires_grad = False

    for epoch in range(num_epochs):
        # Forward pass
        state_inputs = torch.concat([input_values, param.reshape(1, len(initial_params))], dim=1)
        output = m(state_inputs.float())  # Add batch dimension

        # Calculate the loss
        open_loss = criterion(output.float(), target_value.float())
        loss = torch.sqrt(open_loss.sum(axis=1).mean())
        

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history.append((param.tolist(), output.tolist()[0], loss.item()))

        if (epoch + 1) % 100 == 0:
            # print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {param.item():.4f}, Output: {output.item():.4f}')
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {[round(p,4) for p in param.tolist()]}')
        return history


In [6]:
def optimize(row):
    m = model.transition_estimator.state_layer
    inputs = [row.s0, row.s1, row.s2, row.s3, row.a]
    targets = [row.s_0, row.s_1, row.s_2, row.s_3]
    init_param = [row.estimated_p0, row.estimated_p1]

    hist = find_params(m, inputs, targets, init_param)
    return hist[-1]


In [7]:
df[['new_estimated_p', 'new_estimated_s', 'param_rse']] = df.apply(lambda row: optimize(row), axis=1, result_type='expand')

In [8]:
df[['p', 'estimated_p', 'new_estimated_p']].head()

,p,estimated_p,new_estimated_p
0,"(0.5280280669895349, 0.1691488904272192)","(-1.263, -0.169)","[-1.253000020980835, -0.17900000512599945]"
1,"(0.5280280669895349, 0.1691488904272192)","(-1.519, -0.378)","[-1.5090000629425049, -0.3879999816417694]"
2,"(0.5280280669895349, 0.1691488904272192)","(0.975, 0.674)","[0.9650000333786011, 0.6840000152587891]"
3,"(0.5280280669895349, 0.1691488904272192)","(-1.75, -0.48)","[-1.7400000095367432, -0.4899999797344208]"
4,"(0.5280280669895349, 0.1691488904272192)","(0.877, 0.699)","[0.8669999837875366, 0.7089999914169312]"


In [9]:
expansions = {
    'new_estimated_p': ['new_estimated_p0', 'new_estimated_p1'],
}

final_df = data.evaluation_data.copy()
final_df = get_data_expanded(df, expansions)
final_df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s_3,s__0,s__1,s__2,s__3,new_estimated_p,new_estimated_s,param_rse,new_estimated_p0,new_estimated_p1
0,0,0,"(0.5280280669895349, 0.1691488904272192)","(-0.018, -0.046, 0.013, -0.019)",0,1.0,"(-0.019, -0.285, 0.013, 1.258)",0.0,1.0,"(-0.025, -0.525, 0.038, 2.534)",...,1.258,-0.025,-0.525,0.038,2.534,"[-1.253000020980835, -0.17900000512599945]","[0.05903354287147522, -0.2096935510635376, 0.0...",0.727354,-1.253,-0.179
1,1,0,"(0.5280280669895349, 0.1691488904272192)","(-0.019, -0.285, 0.013, 1.258)",0,1.0,"(-0.025, -0.525, 0.038, 2.534)",1.0,1.0,"(-0.035, -0.288, 0.088, 1.326)",...,2.534,-0.035,-0.288,0.088,1.326,"[-1.5090000629425049, -0.3879999816417694]","[-0.02113640308380127, -0.5146619081497192, 0....",0.310064,-1.509,-0.388
2,2,0,"(0.5280280669895349, 0.1691488904272192)","(-0.025, -0.525, 0.038, 2.534)",1,1.0,"(-0.035, -0.288, 0.088, 1.326)",0.0,1.0,"(-0.041, -0.532, 0.115, 2.694)",...,1.326,-0.041,-0.532,0.115,2.694,"[0.9650000333786011, 0.6840000152587891]","[-0.029414385557174683, -0.24926769733428955, ...",0.871338,0.965,0.684
3,3,0,"(0.5280280669895349, 0.1691488904272192)","(-0.035, -0.288, 0.088, 1.326)",0,1.0,"(-0.041, -0.532, 0.115, 2.694)",1.0,1.0,"(-0.052, -0.3, 0.169, 1.601)",...,2.694,-0.052,-0.300,0.169,1.601,"[-1.7400000095367432, -0.4899999797344208]","[-0.03659464418888092, -0.5045229196548462, 0....",0.483155,-1.740,-0.490
4,4,0,"(0.5280280669895349, 0.1691488904272192)","(-0.041, -0.532, 0.115, 2.694)",1,1.0,"(-0.052, -0.3, 0.169, 1.601)",0.0,1.0,"(-0.058, -0.546, 0.201, 3.045)",...,1.601,-0.058,-0.546,0.201,3.045,"[0.8669999837875366, 0.7089999914169312]","[-0.0429975688457489, -0.27913790941238403, 0....",0.679565,0.867,0.709


In [10]:
def predict(row):
    with torch.no_grad():
        m = model.transition_estimator.state_layer
        inputs = [row.s_0, row.s_1, row.s_2, row.s_3, row.a_]
        params = [row.new_estimated_p0, row.new_estimated_p1]
        # params = [row.estimated_p0, row.estimated_p1]

        input_values = torch.tensor(inputs).reshape(1, len(inputs))
        param = torch.tensor(params)

        for p in m.parameters():
            p.requires_grad = False

        state_inputs = torch.concat([input_values, param.reshape(1, len(params))], dim=1)
        output = m(state_inputs.float())  # Add batch dimension

    return [round(v, 3) for v in output.tolist()[0]]


In [11]:
final_df['new_estimated_s'] = final_df.apply(lambda row: predict(row), axis=1)

In [12]:
df_metrics = final_df.copy()
df_metrics['estimated_s'] = df_metrics['new_estimated_s']
results = data.get_evaluation_metrics(df_metrics)

In [13]:
final_df[['p', 'estimated_p', 'new_estimated_p']]

,p,estimated_p,new_estimated_p
0,"(0.5280280669895349, 0.1691488904272192)","(-1.263, -0.169)","[-1.253000020980835, -0.17900000512599945]"
1,"(0.5280280669895349, 0.1691488904272192)","(-1.519, -0.378)","[-1.5090000629425049, -0.3879999816417694]"
2,"(0.5280280669895349, 0.1691488904272192)","(0.975, 0.674)","[0.9650000333786011, 0.6840000152587891]"
3,"(0.5280280669895349, 0.1691488904272192)","(-1.75, -0.48)","[-1.7400000095367432, -0.4899999797344208]"
4,"(0.5280280669895349, 0.1691488904272192)","(0.877, 0.699)","[0.8669999837875366, 0.7089999914169312]"
...,...,...,...
2122,"(0.0368536405463078, 0.7035040297764357)","(0.392, 0.652)","[0.40199998021125793, 0.6420000195503235]"
2123,"(0.0368536405463078, 0.7035040297764357)","(0.33, 0.535)","[0.320000022649765, 0.5450000166893005]"
2124,"(0.0368536405463078, 0.7035040297764357)","(0.406, 0.664)","[0.41599997878074646, 0.6539999842643738]"
2125,"(0.0368536405463078, 0.7035040297764357)","(0.409, 0.775)","[0.4189999997615814, 0.7649999856948853]"


In [14]:
final_df[['s__', 'estimated_s', 'new_estimated_s']]

,s__,estimated_s,new_estimated_s
0,"(-0.025, -0.525, 0.038, 2.534)","(-0.02, -0.516, 0.028, 2.066)","[-0.021, -0.517, 0.028, 2.104]"
1,"(-0.035, -0.288, 0.088, 1.326)","(-0.046, -0.314, 0.113, 0.815)","[-0.049, -0.32, 0.114, 0.839]"
2,"(-0.041, -0.532, 0.115, 2.694)","(-0.001, -0.476, 0.159, 1.341)","[-0.002, -0.48, 0.158, 1.319]"
3,"(-0.052, -0.3, 0.169, 1.601)","(-0.052, -0.282, 0.197, 0.789)","[-0.055, -0.289, 0.197, 0.812]"
4,"(-0.058, -0.546, 0.201, 3.045)","(-0.023, -0.525, 0.215, 1.716)","[-0.024, -0.53, 0.215, 1.692]"
...,...,...,...
2122,"(-0.105, -0.761, 0.128, 0.931)","(-0.115, -0.792, 0.137, 0.924)","[-0.116, -0.796, 0.138, 0.944]"
2123,"(-0.12, -0.952, 0.146, 1.156)","(-0.123, -0.984, 0.142, 1.234)","[-0.124, -0.988, 0.141, 1.197]"
2124,"(-0.139, -1.144, 0.169, 1.383)","(-0.14, -1.144, 0.167, 1.385)","[-0.139, -1.139, 0.168, 1.418]"
2125,"(-0.162, -1.336, 0.197, 1.615)","(-0.162, -1.309, 0.191, 1.585)","[-0.161, -1.304, 0.191, 1.617]"


In [15]:
results[['rse', 'rse_normalized', 
       'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 'rse_r', 'rse_s0_normalized',
       'rse_s1_normalized', 'rse_s2_normalized', 'rse_s3_normalized']].describe()

,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_r,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized
count,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.0,2127.000000,2127.000000,2127.000000,2127.000000
mean,0.122954,0.512904,0.011914,0.021598,0.007876,0.081566,0.0,0.558515,0.531366,0.520086,0.441650
std,0.129194,0.008977,0.014072,0.026202,0.008371,0.114113,0.0,0.014860,0.006441,0.020075,0.009764
min,0.005000,0.502342,0.000000,0.000000,0.000000,0.000000,0.0,0.545935,0.526057,0.501199,0.434671
25%,0.057000,0.507400,0.003000,0.006000,0.003000,0.026000,0.0,0.549102,0.527532,0.508393,0.436896
50%,0.091000,0.510340,0.007000,0.014000,0.006000,0.053000,0.0,0.553326,0.529499,0.515588,0.439206
75%,0.139500,0.515603,0.016000,0.027000,0.010000,0.092000,0.0,0.562830,0.532694,0.525180,0.442543
max,1.692000,0.587200,0.161000,0.330000,0.081000,1.487000,0.0,0.715945,0.607178,0.695444,0.561906


In [16]:
del model
del data